In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# Install dependencies
!apt-get update
!apt-get install -y build-essential cmake git libgtk2.0-dev pkg-config libavcodec-dev libavformat-dev libswscale-dev

# Clone repo (or update if already cloned)
!rm -rf detectron2
!git clone https://github.com/facebookresearch/detectron2.git
%cd detectron2

# Build and install
!python -m pip install -e .


Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.0 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,378 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,910 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,613 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-s

In [ ]:
import detectron2
from detectron2.utils.logger import setup_logger

setup_logger()
print("✅ Detectron2 is installed and ready!")


✅ Detectron2 is installed and ready!


In [ ]:
# ================================
# 5️⃣ Imports
# ================================
import random
import cv2
import matplotlib.pyplot as plt
import torch

from detectron2.engine import DefaultTrainer, DefaultPredictor
from detectron2.config import get_cfg
from detectron2.data import MetadataCatalog, DatasetCatalog
from detectron2.data.datasets import register_coco_instances
from detectron2.utils.visualizer import Visualizer
from detectron2 import model_zoo
from detectron2.data.datasets import register_coco_instances

DATASET_ROOT = "/content/drive/MyDrive/Dataset-COCO-Final/Dataset-COCO-Final"

register_coco_instances(
    "pak_food_train",
    {},
    f"{DATASET_ROOT}/train/_annotations.coco.json",
    f"{DATASET_ROOT}/train"
)

register_coco_instances(
    "pak_food_val",
    {},
    f"{DATASET_ROOT}/val/_annotations.coco.json",
    f"{DATASET_ROOT}/val"
)
from detectron2.config import get_cfg
from detectron2 import model_zoo
import os

CHECKPOINT_DIR = "/content/drive/MyDrive/MaskRCNN-SEGFULL"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

cfg = get_cfg()
cfg.merge_from_file(
    model_zoo.get_config_file(
        "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"
    )
)

# -----------------------------
# Dataset
# -----------------------------
cfg.DATASETS.TRAIN = ("pak_food_train",)
cfg.DATASETS.TEST  = ("pak_food_val",)

# -----------------------------
# Dataloader
# -----------------------------
cfg.DATALOADER.NUM_WORKERS = 2

# -----------------------------
# Model
# -----------------------------
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(
    "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"
)

cfg.MODEL.ROI_HEADS.NUM_CLASSES = 86
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128

cfg.MODEL.DEVICE = "cuda"

# -----------------------------
# Solver (IMPORTANT PART)
# -----------------------------
cfg.SOLVER.IMS_PER_BATCH = 2
cfg.SOLVER.BASE_LR = 0.00025

# 28k images → ~14k iters per epoch
# 8–10 epochs is reasonable
cfg.SOLVER.MAX_ITER = 120_000

cfg.SOLVER.STEPS = (80_000, 100_000)  # LR decay
cfg.SOLVER.GAMMA = 0.1

cfg.SOLVER.CHECKPOINT_PERIOD = 5_000  # 🔑 resume-safe

# -----------------------------
# Output
# -----------------------------
cfg.OUTPUT_DIR = CHECKPOINT_DIR
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)


In [ ]:
import json
import matplotlib.pyplot as plt

METRICS_FILE = "/content/drive/MyDrive/MaskRCNN-SEGFULL/metrics.json"

iters = []
loss_cls = []
loss_box = []
loss_mask = []

with open(METRICS_FILE, "r") as f:
    for line in f:
        d = json.loads(line)
        if all(k in d for k in ["iteration", "loss_cls", "loss_box_reg", "loss_mask"]):
            iters.append(d["iteration"])
            loss_cls.append(d["loss_cls"])
            loss_box.append(d["loss_box_reg"])
            loss_mask.append(d["loss_mask"])

In [ ]:
import json
import matplotlib.pyplot as plt

METRICS_FILE = "/content/drive/MyDrive/MaskRCNN-SEGFULL/metrics.json"

iters = []
loss_cls = []
loss_box = []
loss_mask = []

with open(METRICS_FILE, "r") as f:
    for line in f:
        d = json.loads(line)
        if all(k in d for k in ["iteration", "loss_cls", "loss_box_reg", "loss_mask"]):
            iters.append(d["iteration"])
            loss_cls.append(d["loss_cls"])
            loss_box.append(d["loss_box_reg"])
            loss_mask.append(d["loss_mask"])

In [ ]:
cls_contrib  = loss_cls[0]  - loss_cls[-1]
box_contrib  = loss_box[0]  - loss_box[-1]
mask_contrib = loss_mask[0] - loss_mask[-1]

total = cls_contrib + box_contrib + mask_contrib

cls_pct  = 100 * cls_contrib  / total
box_pct  = 100 * box_contrib  / total
mask_pct = 100 * mask_contrib / total

print(f"📊 Classification Contribution : {cls_pct:.2f}%")
print(f"📦 Bounding Box Contribution   : {box_pct:.2f}%")
print(f"🎯 Segmentation Contribution   : {mask_pct:.2f}%")

📊 Classification Contribution : 83.60%
📦 Bounding Box Contribution   : 6.61%
🎯 Segmentation Contribution   : 9.80%
